In [1]:
import matplotlib.pyplot as plt
import pytorch_lightning as pl
import numpy as np
import torch

from hqm.circuits.angleencoding import BasicEntangledCircuit
from hqm.layers.basiclayer import BasicLayer
from hqm.classification.hcnn import HybridLeNet5
import pennylane as qml

2024-06-21 16:34:05.144474: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-06-21 16:34:05.186866: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-21 16:34:05.186894: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-21 16:34:05.188773: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-06-21 16:34:05.197393: I tensorflow/core/platform/cpu_feature_guar

In [2]:
import sys

In [3]:
# Eseguire lo script download.sh, alla fine ho fatto un file analogo
#!sh C:/Users/danfi/anaconda3/envs/qml/QML-tutorial/download.sh



In [9]:
class HybridNet(pl.LightningModule):

    def __init__(self):
        super(HybridNet, self).__init__()
        dev = qml.device("lightning.qubit", wires=4)
        qcircuit = BasicEntangledCircuit(n_qubits=4, n_layers=2, dev=dev)
        qlayer = BasicLayer(qcircuit, aiframework='torch')
        self.network = HybridLeNet5(qlayer=qlayer, in_shape=(3,64,64), ou_dim=2)
        self.loss = torch.nn.CrossEntropyLoss()

    def forward(self, x):
        return self.network.forward(x)

        
    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)

        #print(labels, outputs)

        loss      = self.loss(outputs, labels)
        # For example, log accuracy
        _, predicted = torch.max(outputs.data, 1)
        accuracy = torch.sum(predicted == labels.data).item() / labels.size(0)

        # Logging info
        self.log('train_loss', loss, on_epoch=True, prog_bar=True)
        self.log('train_accuracy', accuracy, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        
        loss      = self.loss(outputs, labels)
        # For example, log accuracy
        _, predicted = torch.max(outputs.data, 1)
        accuracy = torch.sum(predicted == labels.data).item() / labels.size(0)

        # Logging info
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_accuracy', accuracy, on_epoch=True, prog_bar=True)
        
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.0001)

In [10]:
!pwd

/bin/bash: /home/cappu/anaconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/home/cappu/code/eurosat/QML-tutorial


In [14]:
from pytorch_lightning.callbacks import ModelCheckpoint
import pytorch_lightning as pl
import torch
import sys
import os
import matplotlib.pyplot as plt
import pandas as pd
from tensorboard.backend.event_processing import event_accumulator

from dataio.loader import EuroSATDataModule

In [16]:
from metrics_logger.decorators import metrics_logger as MetricsLogger

#tolgo questa parte perche ho modificato la tecnica di download di eurosat

In [15]:
torch.set_float32_matmul_precision('high')

data_module = EuroSATDataModule(num_workers=16, batch_size=8)

tb_logger = pl.loggers.TensorBoardLogger(os.path.join('lightning_logs','classifiers'), name='EuroSATClassifier')

In [12]:
'''
# Definisco i paths delle mie directory
base_path = "C:/Users/danfi/anaconda3/envs/qml/QML-tutorial"
dataset_path = os.path.join(base_path, "dataset")
train_dir = os.path.join(dataset_path, "training")
val_dir = os.path.join(dataset_path, "validation")

# Ora definisco EuroSATDataModule con i path del mio dataset, EuroSATDataModule è una sottoclasse pl.LightningDataModule
data_module = EuroSATDataModule(train_dir=train_dir, val_dir=val_dir,batch_size=8, num_workers=16)

# definisco TensorBoard logger per registrare i log di addestramento e convalida 
if not os.path.exists(train_dir) or not os.path.exists(val_dir):
    raise FileNotFoundError("Assicurati che le directory del dataset esistano e siano corrette")

# Definisci EuroSATDataModule con i path del dataset
data_module = EuroSATDataModule(train_dir=train_dir, val_dir=val_dir, batch_size=8, num_workers=16)

# Definisci TensorBoard logger per registrare i log di addestramento e convalida
log_dir = os.path.join(base_path, 'lightning_logs', 'classifiers', 'EuroSATClassifier')
tb_logger = pl.loggers.TensorBoardLogger(log_dir)
'''

# Instantiate ModelCheckpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath=os.path.join('saved_models','classifiers'),
    filename='EuroSATClassifier',
    monitor='val_loss',
    save_top_k=1,
    mode='min',
)

In [18]:
#richiamo la funzione che ho fatto io
metrics = MetricsLogger()
# Selezione del dispositivo perche voglio usare cpu
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)
# Instantiate LightningModule and DataModule
model = HybridNet()

model.to(device)


trainer = pl.Trainer(max_epochs=20,callbacks=[checkpoint_callback, metrics], logger=tb_logger, accelerator='gpu')



trainer.fit(model, data_module)
# quindi lancia: tensorboard --logdir=C:/Users/danfi/anaconda3/envs/qml/QML-tutorial/lightning_logs/classifiers/EuroSATClassifier
# Dopo l'allenamento, puoi accedere ai valori memorizzati come segue:

train_losses = metrics_logger.train_losses
val_losses = metrics_logger.val_losses
train_accuracies = metrics_logger.train_accuracies
val_accuracies = metrics_logger.val_accuracies

# Stampa i valori per verificarne il contenuto
print("Train Losses:", train_losses)
print("Validation Losses:", val_losses)
print("Train Accuracies:", train_accuracies)
print("Validation Accuracies:", val_accuracies)

Selected device: cuda:0
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/cappu/anaconda3/envs/esa/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_276746/1823313028.py", line 12, in <module>
    trainer = pl.Trainer(max_epochs=20,callbacks=[checkpoint_callback, metrics], logger=tb_logger, accelerator='gpu')
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cappu/anaconda3/envs/esa/lib/python3.11/site-packages/pytorch_lightning/utilities/argparse.py", line 70, in insert_env_defaults
    return fn(self, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/home/cappu/anaconda3/envs/esa/lib/python3.11/site-packages/pytorch_lightning/trainer/trainer.py", line 430, in __init__
    self._callback_connector.on_trainer_init(
  File "/home/cappu/anaconda3/envs/esa/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/

In [ ]:
#infine provo a plottare il tutto 
# Plotting the metrics
import matplotlib.pyplot as plt

# Supponiamo che i seguenti dati siano stati ottenuti dall'addestramento del tuo modello
epochs = list(range(1, len(train_losses) + 1))

# Plot per Training Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_losses, label='Training Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss vs Epoch')
plt.legend()
plt.grid(True)
plt.show()

# Plot per Validation Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(epochs, val_losses[:len(epochs)], label='Validation Loss', marker='o', color='red')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Validation Loss vs Epoch')
plt.legend()
plt.grid(True)
plt.show()

# Plot per Training Accuracy vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_accuracies, label='Training Accuracy', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Training Accuracy')
plt.title('Training Accuracy vs Epoch')
plt.legend()
plt.grid(True)
plt.show()

# Plot per Validation Accuracy vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(epochs, val_accuracies[:len(epochs)], label='Validation Accuracy', marker='o', color='green')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Validation Accuracy vs Epoch')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
#calcolo max e min accuraratezza/loss
max_train_accuracy = max(train_accuracies)
min_train_loss = min(train_losses)
max_val_accuracy = max(val_accuracies[:len(epochs)])  # Considera solo i valori corrispondenti alle epoche effettive
min_val_loss = min(val_losses[:len(epochs)])  # Considera solo i valori corrispondenti alle epoche effettive

print("Massimo dell'accuratezza di training:", max_train_accuracy)
print("Minimo della loss di training:", min_train_loss)
print("Massimo dell'accuratezza di validazione:", max_val_accuracy)
print("Minimo della loss di validazione:", min_val_loss)